<a href="https://colab.research.google.com/github/giovannasoliveira2021/dados_poluicao/blob/main/dados_poluicao_global_2025_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
!git clone https://github.com/ChiaviniK/dados_polui-o_2526.git

fatal: destination path 'dados_polui-o_2526' already exists and is not an empty directory.


In [13]:
import pandas as pd
import numpy as np

# método de extração - raw
# ETL - Extract - Transform - Load
df_dados_poluicao_raw = pd.read_csv("/content/dados_polui-o_2526/Global_Air_Pollution_Data_2025_2026.csv")

# ver os dados carregados
display(df_dados_poluicao_raw)

,Date,City,Latitude,Longitude,PM2.5,PM10,NO2,SO2,CO,Ozone,Aerosol_Optical_Depth,AQI_Class
0,2025-11-08 00:00:00,"Lahore, Pakistan",31.5497,74.3436,81.0,82.7,94.0,16.4,1910.0,11.0,0.19,Unhealthy
1,2025-11-08 01:00:00,"Lahore, Pakistan",31.5497,74.3436,78.2,79.6,90.9,14.4,1413.0,10.0,0.19,Unhealthy
2,2025-11-08 02:00:00,"Lahore, Pakistan",31.5497,74.3436,75.4,76.8,85.0,12.8,1060.0,12.0,0.19,Unhealthy
3,2025-11-08 03:00:00,"Lahore, Pakistan",31.5497,74.3436,72.2,73.6,72.8,11.8,863.0,20.0,0.19,Unhealthy
4,2025-11-08 04:00:00,"Lahore, Pakistan",31.5497,74.3436,69.8,71.2,57.7,11.1,811.0,31.0,0.19,Unhealthy
...,...,...,...,...,...,...,...,...,...,...,...,...
17467,2026-02-06 19:00:00,"Jakarta, Indonesia",-6.2088,106.8456,49.4,49.9,48.6,39.2,3014.0,5.0,0.16,Unhealthy for Sensitive
17468,2026-02-06 20:00:00,"Jakarta, Indonesia",-6.2088,106.8456,59.6,60.5,47.3,41.6,2791.0,1.0,0.15,Unhealthy
17469,2026-02-06 21:00:00,"Jakarta, Indonesia",-6.2088,106.8456,61.5,62.2,41.5,42.5,2200.0,1.0,0.16,Unhealthy
17470,2026-02-06 22:00:00,"Jakarta, Indonesia",-6.2088,106.8456,61.5,62.2,36.0,42.0,1672.0,1.0,0.16,Unhealthy


In [14]:
# analise preliminar dos dados
# analise de estrutura de dados
df_dados_poluicao_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17472 entries, 0 to 17471
Data columns (total 12 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Date                   17472 non-null  object 
 1   City                   17472 non-null  object 
 2   Latitude               17472 non-null  float64
 3   Longitude              17472 non-null  float64
 4   PM2.5                  17472 non-null  float64
 5   PM10                   17472 non-null  float64
 6   NO2                    17472 non-null  float64
 7   SO2                    17472 non-null  float64
 8   CO                     17472 non-null  float64
 9   Ozone                  17472 non-null  float64
 10  Aerosol_Optical_Depth  17472 non-null  float64
 11  AQI_Class              17472 non-null  object 
dtypes: float64(9), object(3)
memory usage: 1.6+ MB


In [15]:
# resumo estatistico
df_dados_poluicao_raw.describe()

,Latitude,Longitude,PM2.5,PM10,NO2,SO2,CO,Ozone,Aerosol_Optical_Depth
count,17472.000000,17472.000000,17472.000000,17472.000000,17472.000000,17472.000000,17472.000000,17472.000000,17472.000000
mean,23.292375,43.056375,66.060611,69.263124,38.458871,30.292691,1264.659741,54.746337,0.325342
std,23.814690,68.421659,65.414505,66.576392,26.977070,41.777128,1484.401277,49.928268,0.255573
min,-23.550500,-74.006000,0.300000,0.400000,0.000000,0.000000,0.000000,0.000000,0.010000
25%,16.305525,-11.754175,13.175000,14.800000,17.700000,6.100000,278.000000,16.000000,0.120000
50%,30.081800,75.776300,45.500000,48.900000,31.800000,14.400000,713.000000,45.000000,0.270000
75%,40.106350,94.520775,99.700000,104.300000,53.600000,37.900000,1627.000000,71.000000,0.470000
max,51.507400,116.407400,430.300000,434.000000,185.100000,439.300000,13039.000000,288.000000,2.980000


In [19]:
# vamos criar nossa engenharia de dados
def pipeline(df):
  # vamos criar osso backup
  df_limpos = df.copy()
  # sempre qe vc trabalha com BI's ou SQL
  # padronização dos nomes (colocar tudo em snack case)
  df_limpos.columns = [col.lower().replace(' ','_')
  for col in df_limpos.columns]

  # regra de negocio - traduzir numeros para algo legivel
  condicoes = [
      (df_limpos["pm2.5"] <= 12),
      (df_limpos["pm2.5"] <= 34.5),
      (df_limpos["pm2.5"] <= 55.4),
      (df_limpos["pm2.5"] <= 150.4),
      (df_limpos["pm2.5"] <= 150.5)
  ]
  escolha = ["Boa",
             "Moderado",
             "Insaubre para grupos sensíveis",
             "Insalubre",
             "Muito Insalubre/Perigoso"
 ]

 # criar um padrão para os dados
  df_limpos["timestamp"] = pd.to_datetime(df_limpos["date"])

  # agora vamos crar um campo para traduzir os numeros para as escolhas
  df_limpos["status_saude"] = np.select(condicoes, escolha, default="N/A")
  df_limpos["dia"] = df_limpos["timestamp"].dt.day
  df_limpos["mês"] = df_limpos["timestamp"].dt.month
  df_limpos["ano"] = df_limpos["timestamp"].dt.year
  df_limpos["data"] = df_limpos["timestamp"].dt.date
  df_limpos["hora"] = df_limpos["timestamp"].dt.hour

  return df_limpos

In [20]:
df_poluicao_final = pipeline(df_dados_poluicao_raw)
df_poluicao_final.head(50)

,date,city,latitude,longitude,pm2.5,pm10,no2,so2,co,ozone,aerosol_optical_depth,aqi_class,timestamp,status_saude,dia,mês,ano,data,hora
0,2025-11-08 00:00:00,"Lahore, Pakistan",31.5497,74.3436,81.0,82.7,94.0,16.4,1910.0,11.0,0.19,Unhealthy,2025-11-08 00:00:00,Insalubre,8,11,2025,2025-11-08,0
1,2025-11-08 01:00:00,"Lahore, Pakistan",31.5497,74.3436,78.2,79.6,90.9,14.4,1413.0,10.0,0.19,Unhealthy,2025-11-08 01:00:00,Insalubre,8,11,2025,2025-11-08,1
2,2025-11-08 02:00:00,"Lahore, Pakistan",31.5497,74.3436,75.4,76.8,85.0,12.8,1060.0,12.0,0.19,Unhealthy,2025-11-08 02:00:00,Insalubre,8,11,2025,2025-11-08,2
3,2025-11-08 03:00:00,"Lahore, Pakistan",31.5497,74.3436,72.2,73.6,72.8,11.8,863.0,20.0,0.19,Unhealthy,2025-11-08 03:00:00,Insalubre,8,11,2025,2025-11-08,3
4,2025-11-08 04:00:00,"Lahore, Pakistan",31.5497,74.3436,69.8,71.2,57.7,11.1,811.0,31.0,0.19,Unhealthy,2025-11-08 04:00:00,Insalubre,8,11,2025,2025-11-08,4
5,2025-11-08 05:00:00,"Lahore, Pakistan",31.5497,74.3436,66.8,68.2,45.0,11.1,1088.0,43.0,0.20,Unhealthy,2025-11-08 05:00:00,Insalubre,8,11,2025,2025-11-08,5
6,2025-11-08 06:00:00,"Lahore, Pakistan",31.5497,74.3436,71.2,72.6,51.0,11.4,2069.0,36.0,0.20,Unhealthy,2025-11-08 06:00:00,Insalubre,8,11,2025,2025-11-08,6
7,2025-11-08 07:00:00,"Lahore, Pakistan",31.5497,74.3436,80.3,82.0,58.7,11.9,3380.0,28.0,0.21,Unhealthy,2025-11-08 07:00:00,Insalubre,8,11,2025,2025-11-08,7
8,2025-11-08 08:00:00,"Lahore, Pakistan",31.5497,74.3436,79.6,81.0,59.7,13.0,4085.0,34.0,0.21,Unhealthy,2025-11-08 08:00:00,Insalubre,8,11,2025,2025-11-08,8
9,2025-11-08 09:00:00,"Lahore, Pakistan",31.5497,74.3436,71.9,73.1,47.8,15.5,3616.0,68.0,0.20,Unhealthy,2025-11-08 09:00:00,Insalubre,8,11,2025,2025-11-08,9


In [23]:
df_poluicao_final.to_csv("dados_poluicao.csv",
                         index=False,
                         sep=",",
                         encoding="utf-8-sig")

from google.colab import files
files.download("dados_poluicao.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>